# Day 25 - Repartition Coalesce and File Size Mindset

**Topic:** Repartition Coalesce and File Size Mindset  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจความต่างของ `repartition`, `coalesce`, Spark partitions, table partition columns และ file size mindset ก่อนเขียนข้อมูลลง Lakehouse table

วันนี้เราจะฝึกมอง performance ในมุม file layout และ data distribution โดยใช้ PySpark ใน Microsoft Fabric Notebook เพื่อทดลองว่า `repartition()` และ `coalesce()` เปลี่ยนจำนวน Spark partitions อย่างไร และมีผลต่อจำนวนไฟล์ตอน write ลง Lakehouse table ได้อย่างไร

## Goal of the day

1. แยกให้ออกว่า **Spark partition**, **table partition column** และ **output file** ไม่ใช่เรื่องเดียวกัน
2. อธิบายได้ว่า `repartition()` และ `coalesce()` ใช้ต่างกันอย่างไร
3. ใช้ `df.rdd.getNumPartitions()` และ `spark_partition_id()` เพื่อ inspect data distribution เบื้องต้นได้
4. ทดลอง write table หลายรูปแบบ แล้ว inspect metadata เช่น `numFiles`, `sizeInBytes`, `partitionColumns`
5. เริ่มมี file size mindset ว่าทำไม small files, over-partitioning และ `coalesce(1)` แบบสุ่ม ๆ ถึงเป็นปัญหาใน Lakehouse pipeline

## Why it matters in production

ใน production Lakehouse pipeline ปัญหา performance ไม่ได้เกิดจาก transformation logic อย่างเดียว แต่เกิดจาก **file layout** และ **data distribution** ด้วย เช่น:

- write output เป็นไฟล์เล็กจำนวนมาก ทำให้ query และ metadata operation ช้าลง
- repartition มากเกินไป ทำให้เกิด shuffle และสร้าง task/file มากเกินจำเป็น
- coalesce น้อยเกินไป ทำให้ parallelism หาย และบาง task ทำงานหนักเกิน
- partition table ด้วย column ที่มีค่าเยอะเกินไป เช่น `customer_id` อาจทำให้เกิด folder/file กระจายมากเกิน
- ใช้ `coalesce(1)` เพื่อเอาไฟล์เดียวทุกครั้ง ทำให้ pipeline ช้าและไม่ scale

Production takeaway:

> Performance tuning ไม่ใช่แค่ทำให้ code สั้นลง แต่ต้องคิดด้วยว่า data ถูกกระจายอย่างไร และถูกเขียนออกมาเป็นไฟล์แบบไหน

## Key concepts

| Concept | Meaning |
|---|---|
| Spark partition | ส่วนย่อยของข้อมูลภายใน DataFrame/RDD ที่ Spark ใช้แบ่งงานให้ tasks ประมวลผลแบบ parallel |
| Output file | ไฟล์ที่ถูกเขียนออกไปยัง storage เช่น Delta/Parquet file; จำนวน output files มักสัมพันธ์กับจำนวน write tasks แต่ไม่ได้เท่ากันเสมอไป |
| Table partition column | column ที่ใช้แบ่ง physical folder/layout ของ table เช่น `transaction_month`; คนละเรื่องกับ Spark partition |
| `repartition(n)` | กระจาย DataFrame ใหม่เป็น `n` Spark partitions; มักแพงกว่า `coalesce()` เพราะ Spark ต้องย้าย rows ข้าม partitions หรือที่เรียกว่า shuffle |
| `repartition(n, col)` | กระจาย DataFrame ใหม่เป็น `n` Spark partitions โดยใช้ column เป็น key ในการจัด data distribution; มักเกิด shuffle และใช้เมื่ออยากควบคุม data distribution ก่อน write |
| `coalesce(n)` | ลดจำนวน Spark partitions โดยพยายามย้ายข้อมูลให้น้อยกว่า `repartition()`; เหมาะเมื่อข้อมูลหลัง transformation เหลือน้อยลง และต้องการลดจำนวน output files ก่อน write |
| Shuffle | การย้ายข้อมูลข้าม Spark partitions/executors เพื่อจัดกลุ่มหรือกระจายข้อมูลใหม่ มี cost สูงกว่า transformation ที่ไม่ต้องย้ายข้อมูล |
| Small files problem | ปัญหาที่ table/path มีไฟล์เล็กจำนวนมาก ทำให้ read/query ช้าลง และเพิ่ม metadata overhead |
| Over-partitioning | แบ่ง Spark partitions หรือ table partition columns มากเกินไปจนเกิด task/file/folder จำนวนมาก |
| Under-partitioning | แบ่ง Spark partitions น้อยเกินไปจน parallelism ต่ำ และ task บางตัวทำงานหนักเกิน |


## Concept explanation

### Spark partition ไม่ใช่ table partition column

คำว่า partition ใน Spark เจอได้หลายบริบท ต้องแยกให้ชัด:

```python
# Spark partitions inside DataFrame
df_repartitioned = df.repartition(8)

# Table partition column when writing output
df.write.partitionBy("transaction_month").saveAsTable("table_name")
```

สองบรรทัดนี้ไม่เหมือนกัน:

- `repartition(8)` เปลี่ยนจำนวน Spark partitions ก่อนทำงานหรือก่อน write
- `partitionBy("transaction_month")` เปลี่ยน physical layout ของ table ตอน write

> Spark partition = วิธีแบ่งงานประมวลผลแบบ parallel ภายใน Spark    
> Table partition column = วิธีจัด layout ข้อมูลตอนเก็บลง table/storage

### `repartition()` ใช้เมื่อไหร่?

`repartition()` ใช้เมื่อต้องการกระจายข้อมูลใหม่ และมักมี shuffle:

```python
df_repartitioned = df.repartition(8)
df_repartitioned_by_month = df.repartition(8, "transaction_month")
```

เหมาะกับกรณีเช่น:

- ต้องการเพิ่ม parallelism ก่อน write หรือก่อน operation หนัก ๆ
- มี data skew หรือข้อมูลใน Spark partitions ปัจจุบันกระจายไม่สมดุล
- ต้องการจัด data distribution ตาม key บางตัวก่อน write

แต่ข้อควรจำคือ:

> `repartition()` ไม่ฟรี เพราะ shuffle ทำให้ Spark ต้องย้ายข้อมูลและใช้ resource เพิ่ม

### `coalesce()` ใช้เมื่อไหร่?

`coalesce()` มักใช้เพื่อลดจำนวน partitions หลังข้อมูลเล็กลง เช่น หลัง filter หรือหลัง aggregate:

```python
df_smaller = df.filter(F.col("status") == "success")
df_output = df_smaller.coalesce(2)
```

เหมาะกับกรณีเช่น:

- ข้อมูลถูก filter จนเล็กลงมาก
- ต้องการลดจำนวน output files แบบง่าย ๆ
- ต้องการลดจำนวน Spark partitions โดยย้ายข้อมูลให้น้อยกว่า `repartition()`

แต่ไม่ควรใช้แบบสุดโต่ง เช่น:

```python
# Avoid doing this in production
df.coalesce(1)
```

เพราะ `coalesce(1)` ทำให้ DataFrame เหลือ Spark partition เดียว งานช่วงท้ายจึงไม่มี parallelism สำหรับ DataFrame นั้น และอาจกลายเป็น bottleneck ตอน write

### File size mindset

ใน Lab นี้เราใช้ข้อมูลขนาดเล็ก จำนวนและขนาดของไฟล์ที่ได้จึงอาจจะต่างจากสิ่งที่จะเจอในระบบ Production จริงๆ แต่ mindset สำคัญคือ:

- ไฟล์เยอะเกินไป → metadata overhead สูง, query อาจช้า
- ไฟล์ใหญ่เกินไป → parallelism ต่ำ, read อาจกระจายงานได้น้อย
- partition column ละเอียดเกินไป → folder/file กระจัดกระจาย
- write pattern ถี่เกินไป → small files เพิ่มขึ้นเรื่อย ๆ

> เป้าหมายไม่ใช่บังคับให้มีไฟล์จำนวนน้อยที่สุด แต่คือทำให้ file layout เหมาะกับ data volume, query pattern และ maintenance strategy


## Mermaid diagram: Repartition, Coalesce, and File Layout

```mermaid
flowchart LR
    A[Input DataFrame] --> B{Need to change Spark partitions?}
    B -- Increase or redistribute --> C[repartition n or repartition by column]
    B -- Reduce partitions after filter --> D[coalesce n]
    C --> E[Write to Lakehouse table]
    D --> E
    A --> E
    E --> F[Output Delta files]
    E --> G[Optional table partition columns]
    F --> H[File size and file count mindset]
    G --> H
```

Key idea:

> `repartition` และ `coalesce` คุม Spark partitions ก่อน write ส่วน `partitionBy` คุม table/file layout ตอน write ทั้งสามอย่างเกี่ยวข้องกัน แต่ไม่ใช่เรื่องเดียวกัน

## Setup / imports

In [1]:
from datetime import date, timedelta

from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 3, Finished, Available, Finished, False)

## Create mock data

Dataset นี้จำลอง transaction records หลาย region หลายเดือน เพื่อใช้ทดลอง:

- inspect Spark partitions
- repartition / coalesce
- write Delta tables
- write table partitioned by `transaction_month`

In [2]:
# Keep shuffle partitions small for this learning lab.
# This affects shuffle operations such as groupBy, join, distinct, and repartition.
# In production, tune this based on data size, cluster size, and runtime behavior.
spark.conf.set("spark.sql.shuffle.partitions", "8")

regions = ["Bangkok", "Chiang Mai", "Phuket", "Rayong"]
channels = ["web", "mobile", "branch"]
statuses = ["success", "success", "success", "pending", "failed"]
source_systems = ["crm", "payment_gateway", "branch_pos"]
start_date = date(2026, 1, 1)

transactions_data = []

for i in range(1, 241):
    transaction_id = f"T{i:04d}"
    customer_id = 1000 + (i % 60)
    region = regions[i % len(regions)]
    channel = channels[i % len(channels)]
    status = statuses[i % len(statuses)]
    source_system = source_systems[i % len(source_systems)]
    transaction_date = start_date + timedelta(days=i % 90)
    amount = float(((i * 37) % 5000) + 100)

    transactions_data.append((
        transaction_id,
        customer_id,
        region,
        channel,
        amount,
        status,
        source_system,
        transaction_date,
    ))

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("channel", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("transaction_date", T.DateType(), True),
])

# Create an RDD with 4 initial partitions so the starting DataFrame partition count is easy to observe.
transactions_rdd = spark.sparkContext.parallelize(transactions_data, 4)
df_transactions_raw = spark.createDataFrame(transactions_rdd, transaction_schema)

df_transactions = (
    df_transactions_raw
    .withColumn("transaction_month", F.date_format(F.col("transaction_date"), "yyyy-MM"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

df_transactions.show(10, truncate=False)
df_transactions.printSchema()
print("Transaction count:", df_transactions.count())

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 5, Finished, Available, Finished, False)

+--------------+-----------+----------+-------+------+-------+---------------+----------------+-----------------+--------------------------+
|transaction_id|customer_id|region    |channel|amount|status |source_system  |transaction_date|transaction_month|ingestion_timestamp       |
+--------------+-----------+----------+-------+------+-------+---------------+----------------+-----------------+--------------------------+
|T0001         |1001       |Chiang Mai|mobile |137.0 |success|payment_gateway|2026-01-02      |2026-01          |2026-06-13 05:32:12.368288|
|T0002         |1002       |Phuket    |branch |174.0 |success|branch_pos     |2026-01-03      |2026-01          |2026-06-13 05:32:12.368288|
|T0003         |1003       |Rayong    |web    |211.0 |pending|crm            |2026-01-04      |2026-01          |2026-06-13 05:32:12.368288|
|T0004         |1004       |Bangkok   |mobile |248.0 |failed |payment_gateway|2026-01-05      |2026-01          |2026-06-13 05:32:12.368288|
|T0005       

## PySpark code examples

ใน section นี้จะค่อย ๆ ทดลองจาก inspect partition count → ใช้ `repartition()` → ใช้ `coalesce()` → write table → inspect file/table metadata

### Example 1: Inspect current Spark partition count

`df.rdd.getNumPartitions()` ใช้ดูจำนวน Spark partitions ของ DataFrame ในจุดนั้น

> จำนวน partitions ไม่ได้แปลว่าจำนวน output files เท่ากันเสมอ แต่ช่วยให้เข้าใจว่า Spark จะแบ่งงานออกเป็นกี่ส่วนก่อน action/write

In [3]:
def show_partition_count(df, df_name: str) -> None:
    # Print Spark partition count for a DataFrame.
    print(f"{df_name} partitions:", df.rdd.getNumPartitions())


show_partition_count(df_transactions, "df_transactions")

partition_distribution = (
    df_transactions
    .withColumn("spark_partition_id", F.spark_partition_id())
    .groupBy("spark_partition_id")
    .agg(F.count("*").alias("record_count"))
    .orderBy("spark_partition_id")
)

partition_distribution.show()

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 6, Finished, Available, Finished, False)

df_transactions partitions: 4
+------------------+------------+
|spark_partition_id|record_count|
+------------------+------------+
|                 0|          60|
|                 1|          60|
|                 2|          60|
|                 3|          60|
+------------------+------------+



### Example 2: Use `repartition()` to redistribute data

`repartition(8)` จะกระจายข้อมูลใหม่เป็น 8 Spark partitions และมักเกิด shuffle

ใน production ควรถามก่อนว่า:

- ต้องการเพิ่ม parallelism จริงไหม?
- ข้อมูลใหญ่พอให้เพิ่ม partitions แล้วคุ้มไหม?
- shuffle cost คุ้มกับผลลัพธ์ไหม?

In [4]:
df_repartitioned_8 = df_transactions.repartition(8)

show_partition_count(df_transactions, "df_transactions")
show_partition_count(df_repartitioned_8, "df_repartitioned_8")

df_repartitioned_8_distribution = (
    df_repartitioned_8
    .withColumn("spark_partition_id", F.spark_partition_id())
    .groupBy("spark_partition_id")
    .agg(F.count("*").alias("record_count"))
    .orderBy("spark_partition_id")
)

df_repartitioned_8_distribution.show()

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 8, Finished, Available, Finished, False)

df_transactions partitions: 4
df_repartitioned_8 partitions: 8
+------------------+------------+
|spark_partition_id|record_count|
+------------------+------------+
|                 0|          30|
|                 1|          30|
|                 2|          31|
|                 3|          31|
|                 4|          30|
|                 5|          30|
|                 6|          29|
|                 7|          29|
+------------------+------------+



### Example 3: Use `repartition(n, column)` before key-aware output

`repartition(8, "transaction_month")` สั่งให้ Spark shuffle ข้อมูลใหม่ โดยใช้ `transaction_month` เป็น key ในการจัด data distribution

เหมาะสำหรับ lab นี้ เพราะเราจะ write table ที่ partitioned by `transaction_month` ต่อไป

In [7]:
df_repartitioned_by_month = df_transactions.repartition(8, "transaction_month")

show_partition_count(df_repartitioned_by_month, "df_repartitioned_by_month")

month_partition_distribution = (
    df_repartitioned_by_month
    .withColumn("spark_partition_id", F.spark_partition_id())
    .groupBy("spark_partition_id", "transaction_month")
    .agg(F.count("*").alias("record_count"))
    .orderBy("spark_partition_id", "transaction_month")
)

month_partition_distribution.show()

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 14, Finished, Available, Finished, False)

df_repartitioned_by_month partitions: 8
+------------------+-----------------+------------+
|spark_partition_id|transaction_month|record_count|
+------------------+-----------------+------------+
|                 3|          2026-02|          84|
|                 5|          2026-01|          92|
|                 7|          2026-03|          64|
+------------------+-----------------+------------+



### Example 4: Use `coalesce()` to reduce partitions after data becomes smaller

หลังจาก filter แล้วข้อมูลมักเล็กลง การใช้ `coalesce(2)` ช่วยลดจำนวน Spark partitions ก่อน write และช่วยลดจำนวน output files ได้ โดยพยายามย้ายข้อมูลให้น้อยกว่า `repartition()`

Pattern ที่ใช้บ่อย:

```python
df_filtered = df.filter(...)
df_output = df_filtered.coalesce(2)
```

In [8]:
df_success_transactions = (
    df_transactions
    .filter(F.col("status") == "success")
    .filter(F.col("amount") >= 1000)
)

df_success_coalesced_2 = df_success_transactions.coalesce(2)

print("Success transaction count:", df_success_transactions.count())
show_partition_count(df_success_transactions, "df_success_transactions")
show_partition_count(df_success_coalesced_2, "df_success_coalesced_2")

success_partition_distribution = (
    df_success_coalesced_2
    .withColumn("spark_partition_id", F.spark_partition_id())
    .groupBy("spark_partition_id")
    .agg(F.count("*").alias("record_count"))
    .orderBy("spark_partition_id")
)

success_partition_distribution.show()

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 16, Finished, Available, Finished, False)

Success transaction count: 116
df_success_transactions partitions: 4
df_success_coalesced_2 partitions: 2
+------------------+------------+
|spark_partition_id|record_count|
+------------------+------------+
|                 0|          58|
|                 1|          58|
+------------------+------------+



### Example 5: Write default, repartitioned, and coalesced outputs

Cell นี้จะเขียน Lakehouse tables 3 แบบเพื่อเทียบ mindset:

1. default write จาก DataFrame เดิม
2. write หลัง `repartition(8)`
3. write หลัง filter + `coalesce(2)`

In [9]:
table_default = "day25_transactions_default"
table_repartitioned = "day25_transactions_repartitioned_8"
table_coalesced = "day25_success_coalesced_2"

(
    df_transactions
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_default)
)

(
    df_transactions
    .repartition(8)
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_repartitioned)
)

(
    df_success_transactions
    .coalesce(2)
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_coalesced)
)

print("Tables written:")
print("-", table_default)
print("-", table_repartitioned)
print("-", table_coalesced)

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 18, Finished, Available, Finished, False)

Tables written:
- day25_transactions_default
- day25_transactions_repartitioned_8
- day25_success_coalesced_2


### Example 6: Inspect Delta table metadata

ใช้ `DESCRIBE DETAIL` เพื่อดู metadata เบื้องต้น เช่น:

- `format`
- `numFiles`
- `sizeInBytes`
- `partitionColumns`
- `location`

หมายเหตุ: ในงานจริงจำนวน output files อาจไม่ได้เท่ากับจำนวน Spark partitions เสมอไป เพราะ Spark/Delta runtime อาจ optimize การเขียนไฟล์ และผลลัพธ์ยังขึ้นอยู่กับ table partition columns กับขนาดข้อมูลที่เขียนจริง

In [10]:
def show_table_detail(table_name: str) -> None:
    # Show selected DESCRIBE DETAIL columns when available.
    detail_df = spark.sql(f"DESCRIBE DETAIL {table_name}")
    selected_columns = [
        column_name
        for column_name in ["format", "numFiles", "sizeInBytes", "partitionColumns", "location"]
        if column_name in detail_df.columns
    ]

    print(f"DESCRIBE DETAIL: {table_name}")
    detail_df.select(*selected_columns).show(truncate=False)


show_table_detail(table_default)
show_table_detail(table_repartitioned)
show_table_detail(table_coalesced)

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 19, Finished, Available, Finished, False)

DESCRIBE DETAIL: day25_transactions_default
+------+--------+-----------+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------+
|format|numFiles|sizeInBytes|partitionColumns|location                                                                                                                                            |
+------+--------+-----------+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------+
|delta |4       |29166      |[]              |abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day25_transactions_default|
+------+--------+-----------+----------------+------------------------------------------------------------------------------------------------------------------------------

### Example 7: Write table partitioned by `transaction_month`

`partitionBy("transaction_month")` เป็น table/storage layout decision

เหมาะเมื่อ query ส่วนใหญ่ filter ตามเดือน เช่น:

```python
filter(transaction_month == "2026-02")
```

แต่ถ้าเลือก partition column ที่ละเอียดเกินไป เช่น `transaction_id` หรือ `customer_id` อาจกลายเป็น over-partitioning

In [11]:
table_by_month = "day25_transactions_by_month"

(
    df_transactions
    .repartition(8, "transaction_month")
    .write
    .mode("overwrite")
    .format("delta")
    .partitionBy("transaction_month")
    .saveAsTable(table_by_month)
)

print("Partitioned table written:", table_by_month)
show_table_detail(table_by_month)

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 20, Finished, Available, Finished, False)

Partitioned table written: day25_transactions_by_month
DESCRIBE DETAIL: day25_transactions_by_month
+------+--------+-----------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+
|format|numFiles|sizeInBytes|partitionColumns   |location                                                                                                                                             |
+------+--------+-----------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+
|delta |3       |20711      |[transaction_month]|abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day25_transactions_by_month|
+------+--------+-----------+-------------------+---------------------------------------------------

### Example 8: Query with partition column filter

ถ้า table ถูก partition ด้วย `transaction_month` การ query ที่ filter ด้วย column นี้มีโอกาสได้ประโยชน์จาก partition pruning

ใน `.explain(mode="formatted")` ให้สังเกต plan ว่ามี `PartitionFilters` หรือ filter ที่เกี่ยวข้องกับ `transaction_month` ตอน Spark scan table หรือไม่ เพราะสิ่งนี้เป็นสัญญาณว่า Spark สามารถใช้ partition column ช่วยลด data scan ได้

In [12]:
df_feb_transactions = (
    spark.read.table(table_by_month)
    .filter(F.col("transaction_month") == "2026-02")
)

df_feb_summary = (
    df_feb_transactions
    .groupBy("transaction_month", "region")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
    .orderBy("region")
)

df_feb_transactions.explain(mode="formatted")
df_feb_summary.show(truncate=False)

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 21, Finished, Available, Finished, False)

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet spark_catalog.lh_pyspark_challenge.day25_transactions_by_month (1)


(1) Scan parquet spark_catalog.lh_pyspark_challenge.day25_transactions_by_month
Output [10]: [transaction_id#11925, customer_id#11926, region#11927, channel#11928, amount#11929, status#11930, source_system#11931, transaction_date#11932, ingestion_timestamp#11934, transaction_month#11933]
Batched: true
Location: PreparedDeltaFileIndex [abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day25_transactions_by_month]
PartitionFilters: [(transaction_month#11933 = 2026-02)]
ReadSchema: struct<transaction_id:string,customer_id:int,region:string,channel:string,amount:double,status:string,source_system:string,transaction_date:date,ingestion_timestamp:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [10]: [transaction_id#11925, customer_id#11926, region#11927, channel#11928, amou

## Quick recap

| Question | Answer |
|---|---|
| `repartition()` ทำอะไร? | กระจายข้อมูลใหม่เป็นจำนวน Spark partitions ที่กำหนด และมักเกิด shuffle |
| `coalesce()` ทำอะไร? | ลดจำนวน Spark partitions โดยพยายามย้ายข้อมูลให้น้อยกว่า `repartition()` มักใช้เพื่อลดจำนวน output files ก่อน write |
| Spark partition เท่ากับ table partition column ไหม? | ไม่เท่ากัน Spark partition คือการแบ่งงาน ส่วน table partition column คือ storage layout |
| จำนวน partitions เท่ากับจำนวน output files เสมอไหม? | ไม่เสมอไป แต่จำนวน output files มักสัมพันธ์กับจำนวน Spark partitions ตอน write และยังขึ้นอยู่กับ table partition columns, runtime optimization และขนาดข้อมูลที่เขียนจริง |
| `coalesce(1)` ดีไหม? | ไม่ควรใช้แบบสุ่ม ๆ เพราะทำให้ DataFrame เหลือ Spark partition เดียว งานช่วงท้ายจึงไม่มี parallelism และอาจเป็น bottleneck ตอน write |
| partition table ด้วย column แบบไหนที่ควรระวัง? | column ที่ cardinality สูงมาก เช่น `transaction_id`, `customer_id` อาจทำให้เกิด over-partitioning |

## Exercises

### Exercise 1: Compare repartition and coalesce counts

สร้าง DataFrame จาก `df_transactions` แล้วเปรียบเทียบจำนวน Spark partitions

Requirements:

- สร้าง `df_repartitioned_12` ด้วย `repartition(12)`
- สร้าง `df_coalesced_3` จาก `df_repartitioned_12.coalesce(3)`
- ใช้ `show_partition_count()` ตรวจ partition count
- ใช้ `spark_partition_id()` เพื่อดูจำนวน records ต่อ partition

Expected result:

- `df_repartitioned_12` ควรมี 12 Spark partitions
- `df_coalesced_3` ควรเหลือ 3 Spark partitions
- เข้าใจว่า `repartition` เพิ่ม/กระจาย partitions ส่วน `coalesce` ลด partitions

In [13]:
df_repartitioned_12 = df_transactions.repartition(12)
df_coalesced_3 = df_repartitioned_12.coalesce(3)

show_partition_count(df_transactions, "df_transactions")
show_partition_count(df_repartitioned_12, "df_repartitioned_12")
show_partition_count(df_coalesced_3, "df_coalesced_3")

coalesced_distribution = (
    df_coalesced_3
    .withColumn("spark_partition_id", F.spark_partition_id())
    .groupBy("spark_partition_id")
    .agg(F.count("*").alias("record_count"))
    .orderBy("spark_partition_id")
)

coalesced_distribution.show()

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 22, Finished, Available, Finished, False)

df_transactions partitions: 4
df_repartitioned_12 partitions: 12
df_coalesced_3 partitions: 3
+------------------+------------+
|spark_partition_id|record_count|
+------------------+------------+
|                 0|          80|
|                 1|          80|
|                 2|          80|
+------------------+------------+



### Exercise 2: Write a coalesced Gold-style summary table

สร้าง summary รายเดือนจาก `df_transactions` แล้วลด partitions ก่อน write

Requirements:

- group by `transaction_month`, `region`
- aggregate `transaction_count`, `total_amount`, `avg_amount`
- ใช้ `coalesce(2)` ก่อน write
- write table ชื่อ `day25_monthly_region_summary`
- inspect ด้วย `show_table_detail()`

Expected result:

- ได้ table summary ระดับ month-region
- table ถูก write สำเร็จใน Lakehouse
- เห็น metadata เช่น `numFiles` และ `sizeInBytes`

In [14]:
table_monthly_summary = "day25_monthly_region_summary"

df_monthly_region_summary = (
    df_transactions
    .groupBy("transaction_month", "region")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount")
    )
    .orderBy("transaction_month", "region")
)

(
    df_monthly_region_summary
    .coalesce(2)
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_monthly_summary)
)

spark.read.table(table_monthly_summary).show()
show_table_detail(table_monthly_summary)

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 24, Finished, Available, Finished, False)

+-----------------+----------+-----------------+------------+----------+
|transaction_month|    region|transaction_count|total_amount|avg_amount|
+-----------------+----------+-----------------+------------+----------+
|          2026-01|   Bangkok|               23|     55244.0|   2401.91|
|          2026-01|Chiang Mai|               23|     51655.0|   2245.87|
|          2026-01|    Phuket|               24|     55936.0|   2330.67|
|          2026-01|    Rayong|               22|     52670.0|   2394.09|
|          2026-02|   Bangkok|               21|     51736.0|   2463.62|
|          2026-02|Chiang Mai|               21|     56477.0|   2689.38|
|          2026-02|    Phuket|               21|     57254.0|   2726.38|
|          2026-02|    Rayong|               21|     55959.0|   2664.71|
|          2026-03|   Bangkok|               16|     34860.0|   2178.75|
|          2026-03|Chiang Mai|               16|     32048.0|    2003.0|
|          2026-03|    Phuket|               15|   

### Exercise 3: Write and query a region-partitioned table

ทดลอง table partition column อีกแบบด้วย `region`

Requirements:

- write `df_transactions` เป็น table ชื่อ `day25_transactions_by_region`
- ใช้ `.partitionBy("region")`
- ใช้ `.repartition(8, "region")` ก่อน write
- query เฉพาะ `region = "Bangkok"`
- ใช้ `.explain(mode="formatted")` และ `.show()` ตรวจผล

Expected result:

- table มี `partitionColumns = [region]`
- query เฉพาะ Bangkok ได้
- ใน `.explain(mode="formatted")` เห็น `PartitionFilters` ที่เกี่ยวข้องกับ `region`
- เข้าใจว่า table partition column ควรเลือกจาก column ที่มักใช้ filter/query บ่อย ไม่ใช่เลือกสุ่ม ๆ

In [15]:
table_by_region = "day25_transactions_by_region"

(
    df_transactions
    .repartition(8, "region")
    .write
    .mode("overwrite")
    .format("delta")
    .partitionBy("region")
    .saveAsTable(table_by_region)
)

show_table_detail(table_by_region)

df_bangkok_transactions = (
    spark.read.table(table_by_region)
    .filter(F.col("region") == "Bangkok")
)

df_bangkok_transactions.explain(mode="formatted")
df_bangkok_transactions.show(10, truncate=False)
print("Bangkok transaction count:", df_bangkok_transactions.count())

StatementMeta(, c0871586-a8ec-49f7-b56d-68c24b1d606b, 25, Finished, Available, Finished, False)

DESCRIBE DETAIL: day25_transactions_by_region
+------+--------+-----------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|format|numFiles|sizeInBytes|partitionColumns|location                                                                                                                                              |
+------+--------+-----------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|delta |4       |26137      |[region]        |abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day25_transactions_by_region|
+------+--------+-----------+----------------+--------------------------------------------------------------------------------------------------------------------

## Common mistakes

1. **สับสนระหว่าง Spark partition กับ table partition column**

   `df.repartition(8)` คือการจัด Spark partitions ส่วน `.partitionBy("transaction_month")` คือการจัด physical table layout ตอน write

2. **ใช้ `repartition()` แบบเดาสุ่ม**

   `repartition()` มักทำให้เกิด shuffle ถ้าเพิ่มจำนวน partitions โดยไม่มีเหตุผล อาจทำให้ job ช้าลงแทนที่จะเร็วขึ้น

3. **ใช้ `coalesce(1)` ก่อน write ทุกครั้ง**

   การบังคับให้เหลือไฟล์เดียวอาจดูง่ายตอน download แต่ใน production ทำให้ parallelism หายและเกิด bottleneck ได้ง่าย

4. **partition table ด้วย high-cardinality column**

   การ partition ด้วย column เช่น `transaction_id` หรือ `customer_id` มักสร้าง folder/file กระจายมากเกินไป และทำให้ maintenance ยาก

5. **ดูแค่จำนวน files โดยไม่ดู query pattern**

   จำนวนไฟล์น้อยไม่ได้แปลว่าเร็วเสมอ ต้องดู data volume, filter pattern, join pattern และ maintenance strategy ด้วย


## Expected Output / Success Criteria

เมื่อจบ Day 25 ควรทำได้:

- อธิบายความต่างระหว่าง **Spark partition**, **output file** และ **table partition column** ได้
- ใช้ `df.rdd.getNumPartitions()` เพื่อตรวจจำนวน Spark partitions ได้
- ใช้ `F.spark_partition_id()` ร่วมกับ aggregation เพื่อตรวจ distribution ของ records ในแต่ละ Spark partition ได้
- อธิบายได้ว่า `repartition()` มักมี shuffle และใช้เพื่อ redistribute/increase Spark partitions
- อธิบายได้ว่า `coalesce()` เหมาะกับการลด Spark partitions หลังข้อมูลเล็กลง
- write Lakehouse table แบบ default, repartitioned, coalesced และ partitioned by column ได้
- ใช้ `DESCRIBE DETAIL` เพื่อตรวจ metadata เช่น `numFiles`, `sizeInBytes`, `partitionColumns` ได้
- เข้าใจว่าการเลือก partition strategy ต้องสัมพันธ์กับ data volume และ query pattern
- ระวัง small files, over-partitioning, under-partitioning และการใช้ `coalesce(1)` โดยไม่มีเหตุผลชัดเจน

## Final takeaway

Repartition / Coalesce / File Size Mindset คือเรื่องของการคุมสมดุลระหว่าง **parallelism**, **shuffle cost** และ **file layout**

> อย่า tune ด้วยความรู้สึกว่า partitions เยอะหรือน้อยดีกว่าเสมอ ให้ดู data size, query pattern, write pattern และ maintenance cost ร่วมกัน

สำหรับ Data Engineer สิ่งที่ต้องจำคือ:

- `repartition()` ใช้เมื่ออยาก redistribute data แต่ต้องยอมรับ shuffle cost
- `coalesce()` ใช้ลด partitions หลังข้อมูลเล็กลง แต่ไม่ควรลดจนเสีย parallelism
- `partitionBy()` ใช้กำหนด layout ตอนเก็บข้อมูลลง table/storage
- small files เป็นปัญหาเรื่อง metadata/read overhead ไม่ใช่แค่เรื่องความสวยงามของ folder
- ใน production ต้องดู Spark execution plan จาก `.explain()`, ตรวจจำนวนไฟล์และ table partition columns จาก `DESCRIBE DETAIL`, แล้วเทียบกับ query pattern ที่ใช้งานจริง

## Optional cleanup

In [ ]:
# tables_to_drop = [
#     "day25_transactions_default",
#     "day25_transactions_repartitioned_8",
#     "day25_success_coalesced_2",
#     "day25_transactions_by_month",
#     "day25_monthly_region_summary",
#     "day25_transactions_by_region",
# ]

# for table_name in tables_to_drop:
#     spark.sql(f"DROP TABLE IF EXISTS {table_name}")
#     print(f"Dropped table: {table_name}")